# RSNA Knee — Emergency Submission Fallback

This CPU-only notebook creates a structurally valid `submission.csv` using constant 0.5 probabilities. It is a deadline safety fallback only; it is not a trained model and should not be presented as a competitive prediction model.

The notebook reads study IDs from the current competition `test.csv`; Kaggle uses the hidden test set when privately rerunning a submitted notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

COMPETITION_SLUG = 'rsna-knee-abnormality-detection'
TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture',
]
EXPECTED_COLUMNS = ['StudyInstanceUID', *TARGETS]

# Resolve known Kaggle input layouts without broad filesystem searching.
candidate_dirs = [
    Path('/kaggle/input/competitions') / COMPETITION_SLUG,
    Path('/kaggle/input') / COMPETITION_SLUG,
]
competition_dir = next((p for p in candidate_dirs if (p / 'test.csv').is_file()), None)
if competition_dir is None:
    raise FileNotFoundError(f'Could not locate test.csv in: {candidate_dirs}')

test = pd.read_csv(competition_dir / 'test.csv')
if 'StudyInstanceUID' not in test.columns:
    raise ValueError('test.csv does not contain StudyInstanceUID')
study_ids = test['StudyInstanceUID'].astype('string').str.strip()
if len(study_ids) == 0 or study_ids.isna().any() or (study_ids == '').any():
    raise ValueError('test.csv contains no studies or blank study IDs')
if study_ids.duplicated().any():
    raise ValueError('test.csv contains duplicate StudyInstanceUID values')

submission = pd.DataFrame({'StudyInstanceUID': study_ids})
for target in TARGETS:
    submission[target] = 0.5

if submission.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f'Wrong submission columns: {submission.columns.tolist()}')
values = submission[TARGETS].to_numpy(dtype=float)
if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
    raise ValueError('Predictions must be finite probabilities between 0 and 1')

output_path = Path('/kaggle/working/submission.csv')
submission.to_csv(output_path, index=False)
written = pd.read_csv(output_path)
if written.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError('Written submission has an incorrect header')
if written['StudyInstanceUID'].astype(str).tolist() != study_ids.astype(str).tolist():
    raise ValueError('Written study IDs or row order differ from test.csv')
print(f'VALID: {len(written)} studies x {len(TARGETS)} targets')
print(f'Output: {output_path} ({output_path.stat().st_size:,} bytes)')
print(written.head())